In [3]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
from minsearch import Index
client_openai = OpenAI()
from sqlitesearch import TextSearchIndex

In [103]:
messages = [
    {
        'role': 'user', "content" : "I just discovered the course. Can I join it?"
    }
]

response = client_openai.responses.create(
    model = 'gpt-5.4-mini',
    input = messages
)

response.output_text

'Maybe — but I’d need a bit more context to tell you for sure.\n\nIf you mean a specific course, whether you can join usually depends on:\n- whether enrollment is still open,\n- if there are any prerequisites,\n- whether there’s space left,\n- and whether late registration is allowed.\n\nIf you want, send me:\n- the course name,\n- where it’s offered,\n- and the date or term,\n\nand I can help you figure out the next step or draft a message to the instructor/admin asking to join.'

In [4]:
index = TextSearchIndex(
    text_fields=['question', 'answer', 'section'],
    keyword_fields=['course'],
    db_path = 'faq.db'
)

In [10]:
def search(query):
    boost_dict = {'question':3.0, 'section':0.5}
    filter_dict = {'course':'llm-zoomcamp'}

    return index.search(
        query = query,
        boost_dict = boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [9]:
search("how do i run ollama")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npi

In [11]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
response = client_openai.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

NameError: name 'messages' is not defined

In [114]:
call = response.output
call

[ResponseOutputMessage(id='msg_0af25e2e78946cb4006a2a7269a42881a083c78e914f169752', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [108]:
import json

call = response.output[0]
args = json.loads(call.arguments)


args

results = search(**  args)
results_json = json.dumps(results, indent=2)

In [109]:
messages.extend(response.output)
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered course can I join"}', call_id='call_SPV3T14NE0JsoNt15TZEhMP4', name='search', type='function_call', id='fc_0af25e2e78946cb4006a2a7235d51481a0898bb5220cd033a4', namespace=None, status='completed')]

In [110]:
# messages.extend(call)
messages.append(
    {
        'type' : 'function_call_output',
        'call_id' : call.call_id,
        'output' : results_json
    }
)

In [111]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered course can I join"}', call_id='call_SPV3T14NE0JsoNt15TZEhMP4', name='search', type='function_call', id='fc_0af25e2e78946cb4006a2a7235d51481a0898bb5220cd033a4', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_SPV3T14NE0JsoNt15TZEhMP4',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Y

In [112]:
response = client_openai.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

In [113]:
response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [28]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "user", "content": question},
    
]

response = client_openai.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join course"}', call_id='call_ZrhG7SsHZvXePDzC1d68soXy', name='search', type='function_call', id='fc_0e66617d77118388006a2ac1b5c04881a399ef357f13a38423', namespace=None, status='completed')]